# Continuous HI Cosmology Bias Probe Check

This notebook checks the continuous-cosmology HI bias-probe results. It is meant to be readable: each section explains what was trained, what the PCA encoder is doing, how the calibration plot is constructed, and how to interpret the error bars.

Run it from the repo root on Great Lakes, e.g. `/home/jiamingp/diffusion_models_repo`.

## What The Experiment Tests

The question is whether a continuous-cosmology-conditioned diffusion model actually uses the input cosmology, or whether it drifts back toward the training distribution.

- Field: HI only. Mstar and Mtot are not part of this continuous-cosmology run.
- Conditioning: the six CAMELS parameters, in file order: $\Omega_\mathrm{m}$, $\sigma_8$, $A_{\mathrm{SN1}}$, $A_{\mathrm{AGN1}}$, $A_{\mathrm{SN2}}$, $A_{\mathrm{AGN2}}$.
- Architecture: `UNet2DConditionModel` with cross-attention. The normalized parameter vector has shape `(B, 6)` and is passed to the model as `encoder_hidden_states` with shape `(B, 1, 6)`.
- Two regimes: `N=128` 2D training fields for the memorization-regime model, and `N=16384` 2D training fields for the generalization-regime model.
- Held-out cosmologies: fixed simulation indices are excluded from both diffusion training and encoder training. Bias is measured only on generated samples conditioned on these held-out cosmologies.
- Main v1 has CFG off: `cfg_dropout=0.0` during training and `guidance_scale=None` during sampling.

## Training Objective

Each training example is a normalized 2D HI slice `x_0` and a normalized cosmology vector `theta`.

At each step, the scheduler samples a diffusion timestep `t` and Gaussian noise `epsilon`, creates a noisy image `x_t`, and asks the conditional UNet to predict the scheduler target. For these configs the scheduler uses `prediction_type: v_prediction`, squared-cosine betas, and Min-SNR weighting with `min_snr_gamma=5.0`.

Conceptually, the loss is a weighted MSE:

`loss = weight(t) * || UNet(x_t, t, theta) - target_v(x_0, epsilon, t) ||^2`.

The important part for this bias test is that the only information about cosmology is the continuous vector `theta` passed through cross-attention. If the model learns the conditioning, generated fields at a held-out `theta` should encode back to that same `theta`. If not, recovered parameters regress toward the training distribution, and the calibration slope is closer to 0 than 1.

## PCA + Ridge Encoder

The encoder is a diagnostic probe, not part of diffusion training.

Pipeline: normalized HI field -> frozen PCA coefficients -> Ridge head -> six cosmology parameters.

Rules enforced by the scripts:

- PCA is fit on real HI fields only.
- The Ridge head is trained on real HI fields only.
- Held-out cosmologies are excluded before fitting PCA or Ridge.
- Generated fields are never used to fit PCA or the Ridge head.

The PCA basis is only a feature map. The Ridge head is the actual parameter regressor. Before trusting generated-field calibration, we first check the real-data validation MAE and R2 of this encoder.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', Path.cwd())).resolve()
SWEEP_NAME = 'nf_conditional_bias_probe'
RESULT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME
CAL_DIR = RESULT_DIR / 'calibration'
ENC_DIR = RESULT_DIR / 'encoder'

POINTS_PATH = CAL_DIR / 'bias_probe_per_cosmology_points.csv'
SAMPLES_PATH = CAL_DIR / 'bias_probe_per_sample_predictions.csv'
SLOPES_PATH = CAL_DIR / 'bias_probe_regime_slopes.csv'
META_PATH = CAL_DIR / 'bias_probe_eval_metadata.json'
ENCODER_METRICS_PATH = ENC_DIR / 'encoder_val_metrics.csv'
ENCODER_SPLIT_PATH = ENC_DIR / 'encoder_real_split.json'

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 220,
    'font.size': 14,
    'axes.titlesize': 17,
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 23,
})

print('PROJECT_DIR =', PROJECT_DIR)
for p in [POINTS_PATH, SAMPLES_PATH, SLOPES_PATH, META_PATH, ENCODER_METRICS_PATH]:
    print(('found   ' if p.exists() else 'missing '), p)

In [ ]:
def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

points = read_csv_required(POINTS_PATH)
samples = read_csv_required(SAMPLES_PATH)
slopes = read_csv_required(SLOPES_PATH)
metadata = json.loads(META_PATH.read_text()) if META_PATH.exists() else {}

PARAM_DISPLAY_PRETTY = {
    'Omega_m': 'Ωm',
    'sigma_8': 'σ8',
    'A_SN1': 'A_SN1',
    'A_AGN1': 'A_AGN1',
    'A_SN2': 'A_SN2',
    'A_AGN2': 'A_AGN2',
}

def with_parameter_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'parameter' in out.columns:
        out.insert(out.columns.get_loc('parameter') + 1, 'label', out['parameter'].map(PARAM_DISPLAY_PRETTY).fillna(out['parameter']))
    return out

print('point rows:', len(points), 'sample prediction rows:', len(samples), 'slope rows:', len(slopes))
print('PCA basis:', metadata.get('pca_basis_path'))
print('PCA sha256:', metadata.get('pca_basis_sha256'))
print('PCA rank:', metadata.get('pca_rank'), 'explained variance:', metadata.get('pca_explained_variance_sum'))

display(with_parameter_labels(points.head()))
display(with_parameter_labels(slopes.sort_values(['parameter', 'dataset_size'])).round(4))

In [ ]:
if ENCODER_METRICS_PATH.exists():
    enc = pd.read_csv(ENCODER_METRICS_PATH)
    print('Encoder validation metrics. Ωm and σ8 are the most important sanity checks; feedback parameters are expected to be weaker.')
    display(with_parameter_labels(enc.sort_values(['split', 'parameter'])).round(4))
else:
    print('No encoder validation table found:', ENCODER_METRICS_PATH)

if ENCODER_SPLIT_PATH.exists():
    split = json.loads(ENCODER_SPLIT_PATH.read_text())
    keep = {k: split.get(k) for k in ['heldout_indices', 'pca_train_sims', 'head_train_sims', 'head_val_sims'] if k in split}
    print('Encoder split summary:')
    for k, v in keep.items():
        if isinstance(v, list) and len(v) > 12:
            print(k, v[:6], '...', v[-6:], f'(n={len(v)})')
        else:
            print(k, v)

## Cleaner Calibration Plot

Each marker is one held-out input cosmology. The x-axis is the true input parameter. The y-axis is the median recovered parameter after generating many HI samples at that same input cosmology and encoding each generated field back to parameters.

The dashed line is perfect calibration. The fitted line is the main summary: slope near 1 means the generated fields track the input parameter; slope near 0 means the generated fields are nearly independent of the input and regress toward a typical training value.

In [ ]:
PARAM_ORDER = ['Omega_m', 'sigma_8', 'A_SN1', 'A_AGN1', 'A_SN2', 'A_AGN2']
PARAM_LABELS = {
    'Omega_m': r'$\Omega_\mathrm{m}$',
    'sigma_8': r'$\sigma_8$',
    'A_SN1': r'$A_{\mathrm{SN1}}$',
    'A_AGN1': r'$A_{\mathrm{AGN1}}$',
    'A_SN2': r'$A_{\mathrm{SN2}}$',
    'A_AGN2': r'$A_{\mathrm{AGN2}}$',
}
REGIME_LABELS = {'memorization': 'memorization (N=128)', 'generalization': 'generalization (N=16,384)'}
REGIME_COLORS = {'memorization': '#d62728', 'generalization': '#1f77b4'}
REGIME_MARKERS = {'memorization': 'o', 'generalization': 's'}

def plot_clean_calibration(points_df: pd.DataFrame, slopes_df: pd.DataFrame, out_path: Path | None = None) -> None:
    # Keep the main no-CFG run clean. If a future table has multiple guidance labels, filter before calling this function.
    plot_points = points_df.copy()
    if 'guidance_label' in plot_points and plot_points['guidance_label'].nunique() > 1:
        plot_points = plot_points[plot_points['guidance_label'] == 'noguidance'].copy()
    plot_slopes = slopes_df.copy()
    if 'guidance_label' in plot_slopes and plot_slopes['guidance_label'].nunique() > 1:
        plot_slopes = plot_slopes[plot_slopes['guidance_label'] == 'noguidance'].copy()

    fig, axes = plt.subplots(2, 3, figsize=(18.5, 10.5), constrained_layout=True)
    for ax, param in zip(axes.ravel(), PARAM_ORDER):
        sub_param = plot_points[plot_points['parameter'] == param].copy()
        if sub_param.empty:
            ax.set_visible(False)
            continue
        lo = float(min(sub_param['theta_in'].min(), sub_param['theta_rec_q16'].min()))
        hi = float(max(sub_param['theta_in'].max(), sub_param['theta_rec_q84'].max()))
        pad = 0.08 * max(hi - lo, 1e-6)
        lo, hi = lo - pad, hi + pad
        ax.plot([lo, hi], [lo, hi], color='0.25', lw=2.0, ls='--', label='ideal')

        for regime in ['memorization', 'generalization']:
            sub = sub_param[sub_param['regime'] == regime].sort_values('theta_in')
            if sub.empty:
                continue
            y = sub['theta_rec_median'].to_numpy(float)
            yerr = np.vstack([
                y - sub['theta_rec_q16'].to_numpy(float),
                sub['theta_rec_q84'].to_numpy(float) - y,
            ])
            ax.errorbar(
                sub['theta_in'], y, yerr=yerr,
                fmt=REGIME_MARKERS.get(regime, 'o'), ms=7.5, lw=1.8, capsize=3.0,
                color=REGIME_COLORS.get(regime, 'black'), alpha=0.88,
                label=REGIME_LABELS.get(regime, regime),
            )
            fit = plot_slopes[(plot_slopes['parameter'] == param) & (plot_slopes['regime'] == regime)]
            if not fit.empty:
                slope = float(fit['slope'].iloc[0])
                intercept = float(fit['intercept'].iloc[0])
                ax.plot([lo, hi], [slope * lo + intercept, slope * hi + intercept],
                        color=REGIME_COLORS.get(regime, 'black'), lw=3.0)
                ax.text(0.03, 0.93 if regime == 'memorization' else 0.84,
                        f"{REGIME_LABELS[regime]} slope={slope:.2f}",
                        color=REGIME_COLORS[regime], transform=ax.transAxes,
                        fontsize=11.5, ha='left', va='top')

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_title(PARAM_LABELS.get(param, param))
        ax.set_xlabel('Input value')
        ax.set_ylabel('Recovered value')
        ax.grid(alpha=0.18, lw=0.8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    fig.legend(dedup.values(), dedup.keys(), loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.035))
    fig.suptitle('Continuous HI cosmology calibration: recovered vs input', y=1.07)
    if out_path is not None:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, bbox_inches='tight')
        print('wrote', out_path)
    plt.show()

clean_plot_path = CAL_DIR / 'bias_probe_calibration_recovered_vs_input_clean.png'
plot_clean_calibration(points, slopes, clean_plot_path)

## What The Error Bars Mean

For each held-out input cosmology, the sampler generates `K` different HI fields using different noise seeds. Each field is encoded back to recovered parameters. The plotted marker is the median recovered value across those `K` generated samples. The vertical error bar is the 16th to 84th percentile range across those same `K` samples.

So the error bar is not the uncertainty of the input cosmology, and it is not the encoder validation error. It is the stochastic spread of generated samples at fixed input cosmology after passing through the encoder.

If the memorization model has smaller error bars, that usually means it produces less seed-to-seed variation at fixed conditioning. That can happen if it is more collapsed or more tied to memorized training-like fields. Smaller error bars do not automatically mean better calibration; the median location and the fitted slope matter more for bias.

In [ ]:
err = points.copy()
err['half_width_16_84'] = 0.5 * (err['theta_rec_q84'] - err['theta_rec_q16'])
err_summary = (
    err.groupby(['regime', 'dataset_size', 'parameter'], as_index=False)
       .agg(mean_half_width=('half_width_16_84', 'mean'), median_half_width=('half_width_16_84', 'median'))
       .sort_values(['parameter', 'dataset_size'])
)
display(with_parameter_labels(err_summary).round(4))

fig, ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
pivot = err_summary.pivot(index='parameter', columns='regime', values='mean_half_width').reindex(PARAM_ORDER)
x = np.arange(len(pivot))
width = 0.36
ax.bar(x - width/2, pivot.get('memorization'), width, color=REGIME_COLORS['memorization'], label='memorization (N=128)')
ax.bar(x + width/2, pivot.get('generalization'), width, color=REGIME_COLORS['generalization'], label='generalization (N=16,384)')
ax.set_xticks(x, [PARAM_LABELS.get(p, p) for p in pivot.index])
ax.set_ylabel('mean 16-84% half-width')
ax.set_title('Generated-sample spread at fixed input cosmology')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
spread_path = CAL_DIR / 'bias_probe_errorbar_spread_summary.png'
fig.savefig(spread_path, bbox_inches='tight')
print('wrote', spread_path)
plt.show()

## Optional: Compare Another Encoder

The primary encoder should remain PCA + Ridge because it is transparent, real-only, and trained for cosmology recovery. SSCD can be useful as a stress-test feature extractor, but it is trained for natural-image copy detection at 224x224. Since the HI maps are 128x128 scientific fields, SSCD features can be sensitive to resizing and may not be physically calibrated.

If an alternative encoder evaluation writes the same `bias_probe_per_cosmology_points.csv` / `bias_probe_regime_slopes.csv` schema into another directory, set `BIAS_PROBE_ALT_CAL_DIR` and rerun the next cell to compare it with the PCA result.

In [ ]:
alt_dir_env = os.environ.get('BIAS_PROBE_ALT_CAL_DIR', '').strip()
if alt_dir_env:
    ALT_CAL_DIR = Path(alt_dir_env).expanduser().resolve()
    alt_points_path = ALT_CAL_DIR / 'bias_probe_per_cosmology_points.csv'
    alt_slopes_path = ALT_CAL_DIR / 'bias_probe_regime_slopes.csv'
    if alt_points_path.exists() and alt_slopes_path.exists():
        alt_points = pd.read_csv(alt_points_path)
        alt_slopes = pd.read_csv(alt_slopes_path)
        print('Loaded alternative encoder calibration from', ALT_CAL_DIR)
        display(with_parameter_labels(alt_slopes.sort_values(['parameter', 'dataset_size'])).round(4))
        plot_clean_calibration(alt_points, alt_slopes, ALT_CAL_DIR / 'bias_probe_calibration_recovered_vs_input_clean.png')
    else:
        print('Alternative directory was set but expected CSVs were not found:', ALT_CAL_DIR)
else:
    print('No alternative encoder directory set. To compare SSCD or another encoder later, set BIAS_PROBE_ALT_CAL_DIR to its calibration output directory.')

## Interpretation Checklist

Use this order when reading the plot:

1. Check encoder validation first. If the real-data PCA/Ridge encoder cannot recover a parameter on held-out real validation data, generated-field calibration for that parameter is weak evidence.
2. Lead with $\Omega_\mathrm{m}$ and $\sigma_8$; the feedback parameters are usually less tightly constrained by a simple HI-field encoder.
3. Compare fitted slope to the ideal value 1. A flat slope means the generated fields are not strongly tracking the input parameter.
4. Use the vertical bars as generated-sample diversity at fixed input, not as accuracy. Small bars plus biased medians is still bad calibration.
5. Compare memorization vs generalization. The expected result is that the generalization-regime model has a slope closer to 1 for well-constrained parameters.